# Домашнее задание №1 Банковские аккаунты

In [2]:
import csv
import json
from datetime import datetime

TYPO_FIXES = {"diposit": "deposit", "wdraw": "withdraw"}
DATE_FORMATS = ["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"]


def read_file(path):
    """Читает CSV или JSON возвращает список словарей"""
    path = str(path)
    if path.endswith(".csv"):
        with open(path, encoding="utf-8") as file:
            return list(csv.DictReader(file))
    if path.endswith(".json"):
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    raise ValueError("Нужен файл .csv или .json")

## Задание №1 Базовый класс Accoun

In [4]:
class Account:
    """Банковский счёт"""

    _account_counter = 1000
    account_type = "base"
    allowed_operations = ["deposit", "withdraw"]

    def __init__(self, account_holder, balance=0):  # проверяем данные до создания счёта
        self._check_holder(account_holder)
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")

        self.holder = account_holder
        self.account_number = "ACC-" + str(Account._account_counter)
        Account._account_counter += 1
        self._balance = balance
        self.operations_history = []

    def _check_holder(self, name):
        """Нужно в формате «Имя Фамилия», на одном языке"""
        words = name.split()
        if len(words) != 2:
            raise ValueError("Должно состоять из двух слов: Имя Фамилия")

        letters = ""
        for word in words:
            if not word.isalpha():
                raise ValueError("В имени только буквы")
            if not (word[0].isupper() and word[1:].islower()):
                raise ValueError("Каждое слово с заглавной буквы")
            letters += word.lower()

        latin = all("a" <= ch <= "z" for ch in letters)
        cyrillic = all("а" <= ch <= "я" or ch == "ё" for ch in letters)
        if not (latin or cyrillic):
            raise ValueError("Имя только русс. или анг.")

    def _check_amount(self, amount):
        """Сумма > нуля"""
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise ValueError("Сумма положительное число")

   
    def _add_to_history(self, op_type, amount, status, time=None):
        """Запись операции в историю"""
        if time is None:
            time = datetime.now()   
        self.operations_history.append({
            "type": op_type,
            "amount": amount,
            "time": time,
            "balance_after": self._balance, 
            "status": status,
        })

    def deposit(self, amount, time=None):
        """Пополнение счёта"""
        self._check_amount(amount)
        self._balance += amount
        self._add_to_history("deposit", amount, "success", time)

    def withdraw(self, amount, time=None):
        """Снятие денег"""
        self._check_amount(amount)
        if amount > self._balance:  # попытку записываем, со статусом fail
            self._add_to_history("withdraw", amount, "fail", time)
            raise ValueError("Недостаточно средств")
        self._balance -= amount
        self._add_to_history("withdraw", amount, "success", time)

    def get_balance(self):
        """Текущий баланс"""
        return self._balance

    def get_history(self):
        """История операций, копия списка"""
        return list(self.operations_history)

    def show_large_operations(self, n=3):
        """Показывает n последних больших операций"""
        done = [op for op in self.operations_history if op["status"] == "success"]
        if not done:
            print("Операций нет")
            return
        average = sum(op["amount"] for op in done) / len(done)
        large = [op for op in done if op["amount"] >= average]
        large.sort(key=lambda op: op["time"])   # по дате стар потом нов
        print(f"Последние {n} крупных операций (от {average:.2f}):")
        for op in large[-n:]:   #последние n
            print("  ", op["time"], op["type"], op["amount"])

    def _parse_date(self, text):    # нгрязные данные 
        """Перебор форматов даты, не получ. возвр. None"""
        for date_format in DATE_FORMATS:
            try:
                return datetime.strptime(text, date_format)
            except (ValueError, TypeError):
                pass
        return None

    def _clean_row(self, row):
        """Чист. запись что-то не так None."""
        op_type = row["operation"]
        if not op_type:
            return None
        op_type = op_type.strip().lower()
        op_type = TYPO_FIXES.get(op_type, op_type)  # опечатку меняем

        if op_type not in self.allowed_operations:
            return None
        
        try:
            amount = float(row["amount"])   # пусто или текст ошибку дал
        except (ValueError, TypeError):
            return None
        if amount <= 0: # отрицательная сумма
            return None

        time = self._parse_date(row["date"])
        if time is None:
            return None

        return {"type": op_type, "amount": amount, "time": time}

    def clean_history(self, rows, file_number):
        """Оставляет только чистые записи счёта"""
        clean_rows = []
        for row in rows:
            if row["account_number"] != file_number:
                continue 
            clean = self._clean_row(row)
            if clean is not None:   # грязная запись
                clean_rows.append(clean)
        return clean_rows

    def load_history_from_file(self, path, file_number):
        """Загружает операции в счёт и обновляет баланс"""
        rows = read_file(path)
        clean_rows = self.clean_history(rows, file_number)
        clean_rows.sort(key=lambda row: row["time"])   # по дате стар. потом нов.

        for row in clean_rows:
            if row["type"] == "deposit":
                self.deposit(row["amount"], row["time"])
            elif row["type"] == "withdraw":
                try:
                    self.withdraw(row["amount"], row["time"])
                except ValueError:
                    pass
            else:   # готовая сумма в файле
                self._balance += row["amount"]
                self._add_to_history("interest", row["amount"], "success", row["time"])
        return len(clean_rows)

    def __str__(self):
        return f"{self.account_number} [{self.account_type}] {self.holder}: {self._balance:.2f}"

### Проверка задания №1

In [6]:
def try_it(title, action):
    """Проверка на ошибки"""
    try:
        action()
        print(title, "ошибки нет")
    except ValueError as error:
        print(title, " ", error)


acc = Account("Андрей Пупкин", 1000)
acc.deposit(500)
acc.withdraw(200)
print(acc)
for op in acc.get_history():
    print(op)

print()
try_it("Минус на пополнение", lambda: acc.deposit(-50))
try_it("Снять больше баланса", lambda: acc.withdraw(10000))
try_it("Только имя", lambda: Account("Андрей"))
try_it("Имя с маленькой буквы", lambda: Account("андрей пупкин"))
try_it("Смешанные алфавиты", lambda: Account("Andrey Pupkin"))
try_it("Отрицательный баланс", lambda: Account("Андрей Пупкин", -5))

print()
print("Неудачная попытка:")
for op in acc.get_history():
    if op["status"] == "fail":
        print(op)

ACC-1000 [base] Андрей Пупкин: 1300.00
{'type': 'deposit', 'amount': 500, 'time': datetime.datetime(2026, 10, 5, 20, 33, 17, 51401), 'balance_after': 1500, 'status': 'success'}
{'type': 'withdraw', 'amount': 200, 'time': datetime.datetime(2026, 10, 5, 20, 33, 17, 51412), 'balance_after': 1300, 'status': 'success'}

Минус на пополнение   Сумма положительное число
Снять больше баланса   Недостаточно средств
Только имя   Должно состоять из двух слов: Имя Фамилия
Имя с маленькой буквы   Каждое слово с заглавной буквы
Смешанные алфавиты ошибки нет
Отрицательный баланс   Начальный баланс не может быть отрицательным

Неудачная попытка:
{'type': 'withdraw', 'amount': 10000, 'time': datetime.datetime(2026, 10, 5, 20, 33, 17, 51504), 'balance_after': 1300, 'status': 'fail'}


## Задание №2 Наследование

In [8]:
class CheckingAccount(Account):
    """Расчётный счёт работает как базовый."""

    account_type = "checking"
    allowed_operations = ["deposit", "withdraw"]


class SavingsAccount(Account):
    """Сберегательный счёт, можно снять не больше 50%:."""

    account_type = "savings"
    allowed_operations = ["deposit", "withdraw", "interest"]

    def apply_interest(self, rate, time=None):
        """Начисляет проценты на остаток"""
        self._check_amount(rate)
        interest = self._balance * rate / 100
        self._balance += interest
        self._add_to_history("interest", interest, "success", time)

    def withdraw(self, amount, time=None):
        """Снятие: нельзя больше половины баланса."""
        self._check_amount(amount)
        if amount > self._balance * 0.5:
            self._add_to_history("withdraw", amount, "fail", time)
            raise ValueError("Нельзя снять больше 50% баланса")
        super().withdraw(amount, time)

### Проверка задания №2

In [10]:
check = CheckingAccount("Иван Петров", 1000)
save = SavingsAccount("Anna Smith", 5000)

check.deposit(500)
check.withdraw(200)
save.deposit(1000)
save.apply_interest(7)        # +7% на остаток

print(check)
print(save)
print()
print("История сберегательного счёта:")
for op in save.get_history():
    print(op)

print()
try_it("Снять больше 50% со сберегательного", lambda: save.withdraw(4000))
try_it("Снять меньше 50%", lambda: save.withdraw(1000))
print(save)

ACC-1002 [checking] Иван Петров: 1300.00
ACC-1003 [savings] Anna Smith: 6420.00

История сберегательного счёта:
{'type': 'deposit', 'amount': 1000, 'time': datetime.datetime(2026, 10, 5, 20, 33, 17, 67327), 'balance_after': 6000, 'status': 'success'}
{'type': 'interest', 'amount': 420.0, 'time': datetime.datetime(2026, 10, 5, 20, 33, 17, 67336), 'balance_after': 6420.0, 'status': 'success'}

Снять больше 50% со сберегательного   Нельзя снять больше 50% баланса
Снять меньше 50% ошибки нет
ACC-1003 [savings] Anna Smith: 5420.00


### Анализ крупных операций (метод show_large_operations класса Account)

In [12]:
big = CheckingAccount("Мария Волкова")
for amount in (100, 5000, 250, 7000, 300, 9000):
    big.deposit(amount)
big.show_large_operations()

Последние 3 крупных операций (от 3608.33):
   2026-10-05 20:33:17.076456 deposit 5000
   2026-10-05 20:33:17.076458 deposit 7000
   2026-10-05 20:33:17.076460 deposit 9000


## Задание 3 Грязные данные


In [14]:
def find_file(name):
    """Ищет файл  в папке"""
    for folder in ["", "Файлы к ДЗ 1/", "data/", "/Users/artem/Desktop/Файлы к ДЗ 1/"]:
        try:
            open(folder + name).close()
            return folder + name
        except FileNotFoundError:
            pass
    raise FileNotFoundError("Положите " + name + " рядом с ноутом")

ACCOUNTS = [
    ("ACC-100001", CheckingAccount, "Иван Петров", 1200),
    ("ACC-100002", SavingsAccount, "Мария Сидорова", 1500),
    ("ACC-100003", CheckingAccount, "Олег Иванов", 800),
    ("ACC-100004", SavingsAccount, "Елена Кузнецова", 2000),
]


def load_all(file_name):
    """Создаёт четыре счёта и грузит в них историю из файла."""
    path = find_file(file_name)
    print("===", file_name, "===")
    result = []
    for file_number, account_class, holder, start in ACCOUNTS:
        account = account_class(holder, start)
        count = account.load_history_from_file(path, file_number)
        print(file_number, " загружено операций:", count, "|", account)
        result.append(account)
    return result


csv_accounts = load_all("transactions_dirty.csv")
json_accounts = load_all("transactions_dirty.json")

=== transactions_dirty.csv ===
ACC-100001  загружено операций: 20 | ACC-1005 [checking] Иван Петров: 6056.00
ACC-100002  загружено операций: 23 | ACC-1006 [savings] Мария Сидорова: 4004.24
ACC-100003  загружено операций: 14 | ACC-1007 [checking] Олег Иванов: 4959.00
ACC-100004  загружено операций: 25 | ACC-1008 [savings] Елена Кузнецова: 10818.20
=== transactions_dirty.json ===
ACC-100001  загружено операций: 20 | ACC-1009 [checking] Иван Петров: 6056.00
ACC-100002  загружено операций: 23 | ACC-1010 [savings] Мария Сидорова: 4004.24
ACC-100003  загружено операций: 14 | ACC-1011 [checking] Олег Иванов: 4959.00
ACC-100004  загружено операций: 25 | ACC-1012 [savings] Елена Кузнецова: 10818.20


CSV и JSON содержат одни данные поэтому балансы должны совпасть:

In [16]:
for a, b in zip(csv_accounts, json_accounts):
    print(a.holder, round(a.get_balance(), 2), round(b.get_balance(), 2),
          "совпадает" if round(a.get_balance(), 2) == round(b.get_balance(), 2) else "РАЗНЫЕ")

Иван Петров 6056.0 6056.0 совпадает
Мария Сидорова 4004.24 4004.24 совпадает
Олег Иванов 4959.0 4959.0 совпадает
Елена Кузнецова 10818.2 10818.2 совпадает


In [17]:
account = csv_accounts[1]
for op in account.get_history()[:5]:
    print(op["time"], op["type"], op["amount"], op["balance_after"], op["status"])
print()
account.show_large_operations()

2025-09-27 22:17:26 deposit 469.0 1969.0 success
2025-09-27 22:17:26 withdraw 392.0 1577.0 success
2025-09-28 22:17:26 deposit 317.0 1894.0 success
2025-10-02 22:17:26 deposit 840.0 2734.0 success
2025-10-04 22:17:00 interest 183.28 2917.28 success

Последние 3 крупных операций (от 381.58):
   2025-10-09 22:17:26 deposit 622.0
   2025-10-13 22:17:26 withdraw 440.0
   2025-10-22 22:17:26 withdraw 457.0
